# 第十一课:Cross Entropy——大模型的唯一损失函数

讲义:[11-cross-entropy.md](11-cross-entropy.md)。

In [ ]:
import numpy as np

def softmax(x):
    m = np.max(x, axis=-1, keepdims=True)
    e = np.exp(x - m)
    return e / e.sum(-1, keepdims=True)

def cross_entropy(logits, target):
    x = logits - logits.max()                      # 稳定版:对数域一步算完
    return -(x[target] - np.log(np.exp(x).sum()))

for logits, tgt in [([2.0,1.0,0.1], 0), ([5.0,1.0,0.1], 0), ([0.1,2.0,1.0], 0)]:
    logits = np.array(logits)
    print('p =', np.round(softmax(logits), 4), ' CE =', round(cross_entropy(logits, tgt), 4))

## 朴素版 vs 稳定版(讲义 §3、作业 C3)

In [ ]:
z = np.array([1000., 1001, 1002])

with np.errstate(over='ignore', invalid='ignore'):
    naive = -np.log(softmax(z)[1])
print('朴素版 (-log(softmax)):', naive)
print('稳定版:', round(cross_entropy(z, 1), 4))

## batch 版 + label smoothing(讲义 §4–5)

In [ ]:
def cross_entropy_batch(logits, targets):
    x = logits - logits.max(-1, keepdims=True)
    log_p = x - np.log(np.exp(x).sum(-1, keepdims=True))
    return -log_p[np.arange(len(targets)), targets].mean()

rng = np.random.default_rng(5)
L = rng.normal(size=(3, 5))
t = np.array([2, 0, 4])
print('batch 版:', round(cross_entropy_batch(L, t), 6))
print('逐个平均:', round(np.mean([cross_entropy(L[i], t[i]) for i in range(3)]), 6))

# label smoothing
e0 = cross_entropy(L[0], t[0])
p = softmax(L[0])
eps = 0.1
smooth = (1-eps)*(-np.log(p[t[0]])) + eps*np.mean(-np.log(p))
print(f'CE={e0:.4f}  smooth(ε=0.1)={smooth:.4f}')

## 作业现场

- B2:ln(100000) 与「瞎猜 loss」对照。
- D1:第 1 步 loss 异常低的排查思路。

In [ ]:
# TODO(B2): 词表 128,256 的瞎猜 loss
# print(round(np.log(128_256), 2))

# TODO(E1): 三个分布的熵
# for name, p in [('均匀', [1/3]*3), ('中间', [0.7,0.15,0.15]), ('one-hot', [1.,0,0])]:
#     H = -sum(pi*np.log(pi) for pi in p if pi > 0)
#     print(name, round(H, 4))